# Model 3: Generative LLM (advanced/exploratory)

Manually-labeled ABSA data

In this model, full text reviews are fed to a generative LLM which “reads” the reviews and returns aspect-sentiment pairs in a defined JSON format. The OpenAI model `gpt-4.1-mini` was selected because it offers strong performance on structured extraction tasks, reliable instruction following, and significantly lower cost and latency than larger models, making it practical for processing thousands of reviews via the API.

In this notebook, the models are tested on the manually-labeled "gold standard" ABSA data. See `07a` for this model's performance on the data labeled for weak supervision.

## OpenAI API Key

This notebook requires an OpenAI API key.

When running the notebook, you will be prompted to enter your key.

You can obtain a key from:
https://platform.openai.com/api-keys

Approximate cost and time of each API call is reported in the markdown text before the call.

In [1]:
# Install/upgrade the OpenAI Python client
%pip install -U openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Securely request OpenAI API Key from notebook user
import os
from getpass import getpass
from openai import OpenAI

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# Select OpenAI model and prompt lag time
MODEL = "gpt-4.1-mini"
SLEEP_SECONDS = 0.2

Enter your OpenAI API key:  ········


## Imports

In [3]:
import json
import time
import random
import warnings
import pandas as pd
from collections import defaultdict

from tqdm import tqdm
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report
)

warnings.filterwarnings("ignore")

## B. Manual ABSA Evaluation

### B.1. Load and Prepare Manually Labelled ABSA Data

The manually labelled ABSA data is stored in two files: one containing sentence-level review text and metadata, and one containing long-format aspect-sentiment labels. Because labelling was performed at the sentence level, labels are first converted to binary aspect-sentiment columns and then aggregated to the review level so that repeated mentions of the same label within a review are counted only once.

In [4]:
# ---------------------------------------------------------------------------------------
# MANUAL ABSA DATA FILES
# ---------------------------------------------------------------------------------------

MANUAL_TEXT_FILE = "../data/absa_training_set.csv"
MANUAL_LABELS_FILE = "../data/absa_labels_long.csv"

# Save paths
MANUAL_EVAL_OUTFILE = "../data/manual_absa_eval_reviews.csv"
MANUAL_ZERO_RESULTS_OUTFILE = "../data/openai_manual_zero_shot_results.csv"
MANUAL_FEW_RESULTS_OUTFILE = "../data/openai_manual_few_shot_results.csv"
MANUAL_COMPARISON_OUTFILE = "../data/openai_manual_comparison_metrics.csv"

In [5]:
# ---------------------------------------------------------------------------------------
# HELPER MAPS FOR MANUAL LABEL CONVERSION
# ---------------------------------------------------------------------------------------

ASPECT_TO_PREFIX = {
    "Product Quality": "product_quality",
    "Service": "service",
    "Wait Time": "wait_time",
    "Price/Value": "price_value",
    "Cleanliness": "cleanliness",
    "Atmosphere": "atmosphere",
    "General": "general"
}

# Normalization for slightly different style in manual file
ASPECT_NORMALIZATION = {
    "food_quality": "Product Quality",
    "service": "Service",
    "wait_time": "Wait Time",
    "price_value": "Price/Value",
    "cleanliness": "Cleanliness",
    "atmosphere": "Atmosphere",
    "general": "General"
}

SENTIMENT_NORMALIZATION = {
    "positive": "positive",
    "negative": "negative"
}

LABEL_COLS = [
    "product_quality_positive",
    "product_quality_negative",
    "service_positive",
    "service_negative",
    "wait_time_positive",
    "wait_time_negative",
    "price_value_positive",
    "price_value_negative",
    "cleanliness_positive",
    "cleanliness_negative",
    "atmosphere_positive",
    "atmosphere_negative",
    "general_positive",
    "general_negative",
]

In [6]:
# ---------------------------------------------------------------------------------------
# FUNCTION TO BUILD REVIEW-LEVEL MANUAL ABSA EVALUATION SET
# ---------------------------------------------------------------------------------------

def build_manual_absa_review_eval_set(text_file, labels_file, label_cols):
    """
    Build a review-level manual ABSA evaluation set.

    Inputs:
      text_file:
        absa_training_set.csv with columns:
        review_id, gmap_id, rating, sentence_id, sentence_text

      labels_file:
        absa_labels_long.csv with columns:
        review_id, sentence_id, aspect, sentiment

    Output:
      one row per review_id with:
      - full review text reconstructed from ordered sentences
      - review metadata
      - 14 binary gold label columns matching LABEL_COLS

    Notes:
      - repeated labels within a review count once
      - positive/negative labels are aggregated with max()
      - if a review has both positive and negative for the same aspect,
        both binary columns are set to 1
      - reviews with no corresponding valid manual labels are dropped
    """
    # -------------------------------------------------------------------
    # LOAD DATA
    # -------------------------------------------------------------------
    df_text = pd.read_csv(text_file)
    df_labels = pd.read_csv(labels_file)

    # Standardize ids as strings
    for col in ["review_id", "sentence_id"]:
        if col in df_text.columns:
            df_text[col] = df_text[col].astype(str)
        if col in df_labels.columns:
            df_labels[col] = df_labels[col].astype(str)

    if "gmap_id" in df_text.columns:
        df_text["gmap_id"] = df_text["gmap_id"].astype(str)

    # -------------------------------------------------------------------
    # CLEAN / NORMALIZE MANUAL LABELS
    # -------------------------------------------------------------------
    df_labels["aspect"] = (
        df_labels["aspect"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(ASPECT_NORMALIZATION)
    )

    df_labels["sentiment"] = (
        df_labels["sentiment"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(SENTIMENT_NORMALIZATION)
    )

    # Keep only valid rows
    df_labels = df_labels[
        df_labels["aspect"].notna() &
        df_labels["sentiment"].notna()
    ].copy()

    # -------------------------------------------------------------------
    # KEEP ONLY REVIEWS THAT HAVE AT LEAST ONE VALID MANUAL LABEL
    # -------------------------------------------------------------------
    labeled_review_ids = set(df_labels["review_id"].unique())

    df_text = df_text[df_text["review_id"].isin(labeled_review_ids)].copy()

    # -------------------------------------------------------------------
    # CONVERT LONG LABELS TO BINARY REVIEW-LEVEL UPDATES
    # -------------------------------------------------------------------
    label_updates = []

    for _, row in df_labels.iterrows():
        review_id = row["review_id"]
        aspect = row["aspect"]
        sentiment = row["sentiment"]

        if aspect not in ASPECT_TO_PREFIX:
            continue

        prefix = ASPECT_TO_PREFIX[aspect]

        if sentiment == "positive":
            label_updates.append((review_id, f"{prefix}_positive", 1))
        elif sentiment == "negative":
            label_updates.append((review_id, f"{prefix}_negative", 1))

    # -------------------------------------------------------------------
    # BUILD REVIEW-LEVEL GOLD LABEL FRAME
    # -------------------------------------------------------------------
    review_keys = df_text[["review_id"]].drop_duplicates().copy()

    for col in label_cols:
        review_keys[col] = 0

    if len(label_updates) > 0:
        df_updates = pd.DataFrame(
            label_updates,
            columns=["review_id", "label_col", "value"]
        ).drop_duplicates()

        df_updates_wide = (
            df_updates
            .pivot_table(
                index="review_id",
                columns="label_col",
                values="value",
                aggfunc="max",
                fill_value=0
            )
            .reset_index()
        )

        df_gold = review_keys.merge(
            df_updates_wide,
            on="review_id",
            how="left",
            suffixes=("", "__new")
        )

        for col in label_cols:
            new_col = f"{col}__new"
            if new_col in df_gold.columns:
                df_gold[col] = df_gold[[col, new_col]].max(axis=1)
                df_gold.drop(columns=[new_col], inplace=True)
    else:
        df_gold = review_keys.copy()

    # -------------------------------------------------------------------
    # RECONSTRUCT FULL REVIEW TEXT FROM SENTENCES
    # -------------------------------------------------------------------
    df_text = df_text.copy()

    # Try to sort by sentence_id in numeric order when possible
    df_text["_sentence_order"] = pd.to_numeric(df_text["sentence_id"], errors="coerce")
    df_text = df_text.sort_values(["review_id", "_sentence_order", "sentence_id"])

    df_reviews = (
        df_text.groupby("review_id", as_index=False)
        .agg({
            "gmap_id": "first",
            "rating": "first",
            "sentence_text": lambda x: " ".join(
                s.strip() for s in x.dropna().astype(str) if s.strip()
            )
        })
        .rename(columns={"sentence_text": "text"})
    )

    # -------------------------------------------------------------------
    # MERGE REVIEW TEXT + REVIEW-LEVEL GOLD LABELS
    # -------------------------------------------------------------------
    df_manual_eval = df_reviews.merge(df_gold, on="review_id", how="inner")

    for col in label_cols:
        df_manual_eval[col] = df_manual_eval[col].fillna(0).astype(int)

    # -------------------------------------------------------------------
    # SUMMARY
    # -------------------------------------------------------------------
    print("Manual review-level ABSA evaluation set built.")
    print(f"Rows (reviews): {len(df_manual_eval):,}")
    print(f"Unique gmap_id: {df_manual_eval['gmap_id'].nunique():,}")

    print("\nPositive counts per binary label:")
    print(df_manual_eval[label_cols].sum().sort_values(ascending=False))

    return df_manual_eval

In [7]:
# ------------------------------------------------------------
# BUILD MANUAL REVIEW-LEVEL EVALUATION SET
# ------------------------------------------------------------

df_manual_eval = build_manual_absa_review_eval_set(
    text_file=MANUAL_TEXT_FILE,
    labels_file=MANUAL_LABELS_FILE,
    label_cols=LABEL_COLS
)

df_manual_eval.to_csv(MANUAL_EVAL_OUTFILE, index=False)
print(f"\nSaved manual review-level evaluation set to: {MANUAL_EVAL_OUTFILE}")

display(df_manual_eval.head(3))

Manual review-level ABSA evaluation set built.
Rows (reviews): 2,508
Unique gmap_id: 2,404

Positive counts per binary label:
product_quality_positive    708
service_negative            626
general_positive            535
service_positive            529
general_negative            490
product_quality_negative    475
price_value_negative        298
atmosphere_positive         216
price_value_positive        196
wait_time_negative          180
atmosphere_negative         172
cleanliness_negative        127
wait_time_positive          101
cleanliness_positive         76
dtype: int64

Saved manual review-level evaluation set to: ../data/manual_absa_eval_reviews.csv


,review_id,gmap_id,rating,text,product_quality_positive,product_quality_negative,service_positive,service_negative,wait_time_positive,wait_time_negative,price_value_positive,price_value_negative,cleanliness_positive,cleanliness_negative,atmosphere_positive,atmosphere_negative,general_positive,general_negative
0,0003225b6dce3f55ab89db5d8414650bc3509b7ab9be1d...,0x808f902f4d4abbdb:0x41b89c86931d8e59,2,Staff is very rude...the men at the wash only....,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,000b48ff2d981127c7cb6b28cee74ba69a060c91d4be91...,0x80c34d62902150a1:0xcb75062260ee1901,5,Awesome food and excellent service. My first c...,1,0,1,0,0,0,0,0,0,0,0,0,0,0
2,001226755d5e472487797f43f7c9415c4a4b34cc66e27e...,0x80c2da7717e8e873:0x1563378525b796a8,2,"Don't bring kids here. Cockroaches, drug deale...",0,0,0,1,0,0,0,0,0,1,0,0,0,0


### B.2.  Generative LLM Prompt Creation
The same prompt and examples for few-shot approach as the previous `07a` notebook was used.

In [8]:
# -----------------------------------------------------------
# Define Output JSON
# ------------------------------------------------------------

ASPECTS = [
    "Product Quality",
    "Service",
    "Wait Time",
    "Price/Value",
    "Cleanliness",
    "Atmosphere",
    "General"
]

SENTIMENTS = ["positive", "negative", "neutral", "mixed"]

ABSA_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "absa_review_output",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "review_id": {"type": "string"},
                "pairs": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "aspect": {"type": "string", "enum": ASPECTS},
                            "sentiment": {"type": "string", "enum": SENTIMENTS},
                            "evidence": {"type": "string"}
                        },
                        "required": ["aspect", "sentiment", "evidence"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["review_id", "pairs"],
            "additionalProperties": False
        }
    }
}

In [9]:
# ----------------------------------------------------------------------------
# Define Prompt
# ----------------------------------------------------------------------------

DEVELOPER_PROMPT = """
You are an expert annotator for aspect-based sentiment analysis (ABSA).

Task:
Read one customer review and extract all aspect-sentiment pairs present.

Allowed aspects:
- Product Quality
- Service
- Wait Time
- Price/Value
- Cleanliness
- Atmosphere
- General

Allowed sentiment labels:
- positive
- negative
- neutral
- mixed

Rules:
1. Return only the requested JSON schema.
2. Include one entry per distinct aspect mentioned in the review.
3. Do not invent aspects not supported by the review.
4. Use General if there is an overall sentiment that does not clearly fit another aspect.
5. Keep evidence short and directly grounded in the review text.
6. If an aspect contains both positive and negative sentiment, use mixed.
7. If there is no clear aspect sentiment, return an empty list for pairs.
"""

In [10]:
# --------------------------------------------------------------------------
# FEW-SHOT EXAMPLES
# --------------------------------------------------------------------------

FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": "Review ID: ex1\nReview: The tacos were amazing and cheap, but the dining room was dirty."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex1",
            "pairs": [
                {"aspect": "Product Quality", "sentiment": "positive", "evidence": "tacos were amazing"},
                {"aspect": "Price/Value", "sentiment": "positive", "evidence": "cheap"},
                {"aspect": "Cleanliness", "sentiment": "negative", "evidence": "dining room was dirty"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex2\nReview: Our server was kind, but we waited 40 minutes for the food."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex2",
            "pairs": [
                {"aspect": "Service", "sentiment": "positive", "evidence": "server was kind"},
                {"aspect": "Wait Time", "sentiment": "negative", "evidence": "waited 40 minutes"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex3\nReview: Great place overall. I'll definitely come back."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex3",
            "pairs": [
                {"aspect": "General", "sentiment": "positive", "evidence": "Great place overall"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex4\nReview: The restaurant looked nice but the tables were sticky."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex4",
            "pairs": [
                {"aspect": "Atmosphere", "sentiment": "positive", "evidence": "restaurant looked nice"},
                {"aspect": "Cleanliness", "sentiment": "negative", "evidence": "tables were sticky"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex5\nReview: The burger was okay and the price was fair."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex5",
            "pairs": [
                {"aspect": "Product Quality", "sentiment": "neutral", "evidence": "burger was okay"},
                {"aspect": "Price/Value", "sentiment": "neutral", "evidence": "price was fair"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex6\nReview: The staff were friendly but also forgot our drinks several times."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex6",
            "pairs": [
                {"aspect": "Service", "sentiment": "mixed", "evidence": "staff were friendly but forgot our drinks"}
            ]
        })
    },
    {
        "role": "user",
        "content": "Review ID: ex7\nReview: The line moved quickly and the food came out fast."
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "review_id": "ex7",
            "pairs": [
                {"aspect": "Wait Time", "sentiment": "positive", "evidence": "line moved quickly"}
            ]
        })
    }
]

### B.3. Build Prompts

Same functions as `07a` notebook were made to build the input to the LLM (prompt + review) and call the OpenAI API in oder to pass the prompt and request the desired JSON output.

In [11]:
# Function to build prompt
def build_messages(review_id, review_text, mode="zero_shot"):
    messages = [{"role": "developer", "content": DEVELOPER_PROMPT}]

    if mode == "few_shot":
        messages.extend(FEW_SHOT_EXAMPLES)

    messages.append({
        "role": "user",
        "content": f"Review ID: {review_id}\nReview: {review_text}"
    })

    return messages

In [12]:
# Function to call API
def call_absa_model(review_id, review_text, mode="zero_shot", max_retries=3):
    """
    Send one review to OpenAI and return parsed structured output.
    Also returns token usage for later cost analysis.
    """
    messages = build_messages(review_id, review_text, mode=mode)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                response_format=ABSA_SCHEMA,
                temperature=0
            )

            content = response.choices[0].message.content
            parsed = json.loads(content)
            usage = response.usage

            if "review_id" not in parsed:
                parsed["review_id"] = str(review_id)
            if "pairs" not in parsed or parsed["pairs"] is None:
                parsed["pairs"] = []

            return {
                "review_id": str(review_id),
                "success": True,
                "parsed": parsed,
                "error": None,
                "prompt_tokens": getattr(usage, "prompt_tokens", None),
                "completion_tokens": getattr(usage, "completion_tokens", None),
                "total_tokens": getattr(usage, "total_tokens", None),
            }

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)
            else:
                return {
                    "review_id": str(review_id),
                    "success": False,
                    "parsed": {"review_id": str(review_id), "pairs": []},
                    "error": str(e),
                    "prompt_tokens": None,
                    "completion_tokens": None,
                    "total_tokens": None,
                }

In [13]:
# Map API output to 14 ABSA labels

ASPECT_TO_PREFIX = {
    "Product Quality": "product_quality",
    "Service": "service",
    "Wait Time": "wait_time",
    "Price/Value": "price_value",
    "Cleanliness": "cleanliness",
    "Atmosphere": "atmosphere",
    "General": "general"
}

def pairs_to_binary_label_dict(pairs):
    """
    Convert ABSA pairs into the same 14 binary columns as the weak ABSA labels.
    """
    out = {col: 0 for col in LABEL_COLS}

    for pair in pairs:
        aspect = pair.get("aspect")
        sentiment = pair.get("sentiment")

        if aspect not in ASPECT_TO_PREFIX:
            continue

        prefix = ASPECT_TO_PREFIX[aspect]

        if sentiment == "positive":
            out[f"{prefix}_positive"] = 1
        elif sentiment == "negative":
            out[f"{prefix}_negative"] = 1
        elif sentiment == "mixed":
            out[f"{prefix}_positive"] = 1
            out[f"{prefix}_negative"] = 1
        elif sentiment == "neutral":
            pass

    return out

### B.4. Prompt model via API over entire dataframe
Same function as `07a` was used to call the OpenAI API on the entire dataframe of manually labeled reviews.

In [14]:
# ------------------------------------------------------------
# FUNCTION FOR MODEL PROMPTING VIA API
# ------------------------------------------------------------

def run_openai_absa(df_input, mode="zero_shot", sleep_seconds=0.2, save_every=100, outfile=None):
    """
    Run OpenAI ABSA extraction over a dataframe and return a dataframe
    with predicted binary label columns added.

    If outfile is provided, partial checkpoints are saved every `save_every` rows.
    """
    results = []

    for idx, (_, row) in enumerate(tqdm(df_input.iterrows(), total=len(df_input)), start=1):
        review_id = str(row["review_id"])
        review_text = str(row["text"])

        result = call_absa_model(
            review_id=review_id,
            review_text=review_text,
            mode=mode,
            max_retries=3
        )

        pred_dict = pairs_to_binary_label_dict(result["parsed"].get("pairs", []))

        out_row = {
            "review_id": review_id,
            "rating": row["rating"],
            "text": row["text"],
            "gmap_id": row["gmap_id"],
            "mode": mode,
            "success": result["success"],
            "error": result["error"],
            "raw_pairs": json.dumps(result["parsed"].get("pairs", [])),
            "prompt_tokens": result["prompt_tokens"],
            "completion_tokens": result["completion_tokens"],
            "total_tokens": result["total_tokens"],
        }

        for col in LABEL_COLS:
            out_row[f"true__{col}"] = int(row[col])

        for col in LABEL_COLS:
            out_row[f"pred__{col}"] = int(pred_dict[col])

        results.append(out_row)

        if outfile is not None and idx % save_every == 0:
            pd.DataFrame(results).to_csv(outfile, index=False)
            print(f"\nCheckpoint saved: {outfile} ({idx:,} rows processed)")

        if sleep_seconds > 0:
            time.sleep(sleep_seconds)

    results_df = pd.DataFrame(results)

    if outfile is not None:
        results_df.to_csv(outfile, index=False)
        print(f"\nFinal saved: {outfile}")

    return results_df

### B.5. Run Zero-Shot on the Manual ABSA Review Set

Zero-shot prompting is first applied to the manually labelled review set. Each reconstructed full review is sent to the model, and the returned aspect-sentiment pairs are mapped into the same 14 binary label columns used for evaluation.


In [15]:
results_manual_zero = run_openai_absa(
    df_input=df_manual_eval,
    mode="zero_shot",
    sleep_seconds=SLEEP_SECONDS,
    save_every=100,
    outfile=MANUAL_ZERO_RESULTS_OUTFILE
)

  4%|▍         | 100/2508 [03:33<1:15:56,  1.89s/it]


Checkpoint saved: ../data/openai_manual_zero_shot_results.csv (100 rows processed)


  8%|▊         | 200/2508 [07:02<1:36:24,  2.51s/it]


Checkpoint saved: ../data/openai_manual_zero_shot_results.csv (200 rows processed)


 12%|█▏        | 300/2508 [10:12<1:04:47,  1.76s/it]

...


Checkpoint saved: ../data/openai_manual_zero_shot_results.csv (2,400 rows processed)


100%|█████████▉| 2500/2508 [1:17:44<00:13,  1.71s/it]


Checkpoint saved: ../data/openai_manual_zero_shot_results.csv (2,500 rows processed)


100%|██████████| 2508/2508 [1:17:58<00:00,  1.87s/it]


Final saved: ../data/openai_manual_zero_shot_results.csv


### B.6. Run Few-Shot on the Manual ABSA Review Set

Few-shot prompting is then applied to the same manually labelled review set using the example review-response pairs defined earlier.

In [16]:
results_manual_few = run_openai_absa(
    df_input=df_manual_eval,
    mode="few_shot",
    sleep_seconds=SLEEP_SECONDS,
    save_every=100,
    outfile=MANUAL_FEW_RESULTS_OUTFILE
)

  4%|▍         | 100/2508 [02:42<1:01:45,  1.54s/it]


Checkpoint saved: ../data/openai_manual_few_shot_results.csv (100 rows processed)


  8%|▊         | 200/2508 [05:40<1:10:39,  1.84s/it]


Checkpoint saved: ../data/openai_manual_few_shot_results.csv (200 rows processed)


 12%|█▏        | 300/2508 [08:25<58:48,  1.60s/it]  

...


Checkpoint saved: ../data/openai_manual_few_shot_results.csv (2,400 rows processed)


100%|█████████▉| 2500/2508 [1:14:37<00:13,  1.74s/it]


Checkpoint saved: ../data/openai_manual_few_shot_results.csv (2,500 rows processed)


100%|██████████| 2508/2508 [1:14:53<00:00,  1.79s/it]


Final saved: ../data/openai_manual_few_shot_results.csv


### B.7. Evaluate Manual ABSA Results

The zero-shot and few-shot outputs are evaluated against the manually labelled review-level gold labels using subset accuracy, macro F1, weighted F1, and a full classification report.

In [17]:
# -----------------------------------------------------------
# Function for multi-label evaluation of model performance
# ------------------------------------------------------------

def evaluate_multilabel_results(results_df, label_cols):
    """
    Compute:
    - weighted precision
    - weighted recall
    - weighted F1
    - macro F1
    - subset accuracy
    - classification report
    """
    true_cols = [f"true__{c}" for c in label_cols]
    pred_cols = [f"pred__{c}" for c in label_cols]

    y_true = results_df[true_cols].values
    y_pred = results_df[pred_cols].values

    metrics = {
        "precision": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
        "classification_report": classification_report(
            y_true,
            y_pred,
            target_names=label_cols,
            zero_division=0
        )
    }

    return metrics


def print_multilabel_results(model_name, metrics):
    """
    Print results in a format that matches your other model output.
    """
    print(f"{model_name}")
    print(f"Test Accuracy:     {metrics['accuracy']:.4f}\n")
    print(f"F1 Score (macro):    {metrics['macro_f1']:.4f}")
    print(f"F1 Score (weighted): {metrics['weighted_f1']:.4f}\n")
    print("Classification Report:")
    print(metrics["classification_report"])

In [18]:
manual_zero_metrics = evaluate_multilabel_results(results_manual_zero, LABEL_COLS)
manual_few_metrics = evaluate_multilabel_results(results_manual_few, LABEL_COLS)

print_multilabel_results("MANUAL ABSA ZERO-SHOT METRICS", manual_zero_metrics)
print()
print_multilabel_results("MANUAL ABSA FEW-SHOT METRICS", manual_few_metrics)

MANUAL ABSA ZERO-SHOT METRICS
Test Accuracy:     0.4510

F1 Score (macro):    0.7522
F1 Score (weighted): 0.7612

Classification Report:
                          precision    recall  f1-score   support

product_quality_positive       0.81      0.84      0.82       708
product_quality_negative       0.73      0.81      0.77       475
        service_positive       0.86      0.96      0.91       529
        service_negative       0.83      0.93      0.87       626
      wait_time_positive       0.69      0.58      0.63       101
      wait_time_negative       0.65      0.80      0.72       180
    price_value_positive       0.80      0.89      0.85       196
    price_value_negative       0.78      0.89      0.83       298
    cleanliness_positive       0.86      0.95      0.90        76
    cleanliness_negative       0.75      0.77      0.76       127
     atmosphere_positive       0.65      0.79      0.71       216
     atmosphere_negative       0.70      0.52      0.59       172
    

### B.5. Compare Zero-Shot vs. Few-Shot on Manual ABSA Set

In [19]:
# ------------------------------------------------------------
# FUNCTION TO CONVERT METRICS DICT TO A CLEAN COMPARISON ROW
# ------------------------------------------------------------

def metrics_to_row(mode_name, metrics):
    return {
        "mode": mode_name,
        "accuracy": metrics["accuracy"],
        "precision_weighted": metrics["precision"],
        "recall_weighted": metrics["recall"],
        "f1_macro": metrics["macro_f1"],
        "f1_weighted": metrics["weighted_f1"],
    }
manual_comparison = pd.DataFrame([
    metrics_to_row("zero_shot", manual_zero_metrics),
    metrics_to_row("few_shot", manual_few_metrics),
])

display(manual_comparison)

manual_comparison.to_csv(MANUAL_COMPARISON_OUTFILE, index=False)
print(f"\nSaved manual comparison metrics to: {MANUAL_COMPARISON_OUTFILE}")

,mode,accuracy,precision_weighted,recall_weighted,f1_macro,f1_weighted
0,zero_shot,0.450957,0.747266,0.781772,0.752169,0.761170
1,few_shot,0.418660,0.784977,0.710510,0.731246,0.741032



Saved manual comparison metrics to: ../data/openai_manual_comparison_metrics.csv


### B.6. Token Usage Summary for Manual ABSA Set

In [20]:
# -----------------------------------------------------------
# Function to calculate token usage
# ------------------------------------------------------------

def summarize_usage(results_df, mode_name):
    prompt_tokens = pd.to_numeric(results_df["prompt_tokens"], errors="coerce").fillna(0).sum()
    completion_tokens = pd.to_numeric(results_df["completion_tokens"], errors="coerce").fillna(0).sum()
    total_tokens = pd.to_numeric(results_df["total_tokens"], errors="coerce").fillna(0).sum()

    print(f"\n{mode_name} token usage")
    print(f"  prompt_tokens:     {int(prompt_tokens):,}")
    print(f"  completion_tokens: {int(completion_tokens):,}")
    print(f"  total_tokens:      {int(total_tokens):,}")

In [21]:
summarize_usage(results_manual_zero, "MANUAL ZERO-SHOT")
summarize_usage(results_manual_few, "MANUAL FEW-SHOT")


MANUAL ZERO-SHOT token usage
  prompt_tokens:     929,413
  completion_tokens: 228,708
  total_tokens:      1,158,121

MANUAL FEW-SHOT token usage
  prompt_tokens:     2,238,589
  completion_tokens: 208,306
  total_tokens:      2,446,895


In [22]:
# ------------------------------------------------------------
# REVIEW-LEVEL ERROR ANALYSIS
# Pull best / worst reviews based on row-level multilabel F1
# ------------------------------------------------------------

def add_row_level_scores(results_df, label_cols):
    """
    Add row-level precision, recall, F1, and exact-match flag.

    For each review:
    - precision = TP / (TP + FP)
    - recall    = TP / (TP + FN)
    - f1        = harmonic mean of precision and recall
    - exact_match = 1 if all labels match exactly, else 0
    """
    df = results_df.copy()

    true_cols = [f"true__{c}" for c in label_cols]
    pred_cols = [f"pred__{c}" for c in label_cols]

    row_precisions = []
    row_recalls = []
    row_f1s = []
    row_exact = []
    row_tp = []
    row_fp = []
    row_fn = []

    for _, row in df.iterrows():
        y_true = row[true_cols].astype(int).values
        y_pred = row[pred_cols].astype(int).values

        tp = ((y_true == 1) & (y_pred == 1)).sum()
        fp = ((y_true == 0) & (y_pred == 1)).sum()
        fn = ((y_true == 1) & (y_pred == 0)).sum()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0
        exact_match = int((y_true == y_pred).all())

        row_tp.append(tp)
        row_fp.append(fp)
        row_fn.append(fn)
        row_precisions.append(precision)
        row_recalls.append(recall)
        row_f1s.append(f1)
        row_exact.append(exact_match)

    df["row_tp"] = row_tp
    df["row_fp"] = row_fp
    df["row_fn"] = row_fn
    df["row_precision"] = row_precisions
    df["row_recall"] = row_recalls
    df["row_f1"] = row_f1s
    df["exact_match"] = row_exact

    return df


def get_true_pred_label_lists(row, label_cols):
    """
    Return lists of active true and predicted labels for one row.
    """
    true_labels = [col for col in label_cols if int(row[f"true__{col}"]) == 1]
    pred_labels = [col for col in label_cols if int(row[f"pred__{col}"]) == 1]
    return true_labels, pred_labels


def build_error_analysis_table(scored_df, label_cols, n=5, best=True):
    """
    Create a compact table of the best or worst rows.

    Ranking:
    - Best: highest row_f1, then highest exact_match, then lowest total mistakes
    - Worst: lowest row_f1, then lowest exact_match, then highest total mistakes
    """
    df = scored_df.copy()

    df["total_errors"] = df["row_fp"] + df["row_fn"]

    if best:
        df_ranked = df.sort_values(
            by=["row_f1", "exact_match", "total_errors"],
            ascending=[False, False, True]
        ).head(n)
    else:
        df_ranked = df.sort_values(
            by=["row_f1", "exact_match", "total_errors"],
            ascending=[True, True, False]
        ).head(n)

    rows = []
    for _, row in df_ranked.iterrows():
        true_labels, pred_labels = get_true_pred_label_lists(row, label_cols)

        rows.append({
            "review_id": row["review_id"],
            "rating": row.get("rating", None),
            "row_f1": round(row["row_f1"], 4),
            "row_precision": round(row["row_precision"], 4),
            "row_recall": round(row["row_recall"], 4),
            "exact_match": row["exact_match"],
            "tp": row["row_tp"],
            "fp": row["row_fp"],
            "fn": row["row_fn"],
            "text": row.get("text", ""),
            "true_labels": ", ".join(true_labels),
            "pred_labels": ", ".join(pred_labels),
            "raw_pairs": row.get("raw_pairs", "")
        })

    return pd.DataFrame(rows)

In [23]:
# ------------------------------------------------------------
# ADD ROW-LEVEL SCORES
# ------------------------------------------------------------

scored_manual_zero = add_row_level_scores(results_manual_zero, LABEL_COLS)
scored_manual_few = add_row_level_scores(results_manual_few, LABEL_COLS)

print("Zero-shot scored shape:", scored_manual_zero.shape)
print("Few-shot scored shape:", scored_manual_few.shape)

Zero-shot scored shape: (2508, 46)
Few-shot scored shape: (2508, 46)


In [24]:
# ------------------------------------------------------------
# BUILD BEST / WORST ERROR ANALYSIS TABLES
# ------------------------------------------------------------

manual_zero_best = build_error_analysis_table(
    scored_df=scored_manual_zero,
    label_cols=LABEL_COLS,
    n=10,
    best=True
)

manual_zero_worst = build_error_analysis_table(
    scored_df=scored_manual_zero,
    label_cols=LABEL_COLS,
    n=10,
    best=False
)

manual_few_best = build_error_analysis_table(
    scored_df=scored_manual_few,
    label_cols=LABEL_COLS,
    n=10,
    best=True
)

manual_few_worst = build_error_analysis_table(
    scored_df=scored_manual_few,
    label_cols=LABEL_COLS,
    n=10,
    best=False
)

In [25]:
# Make long review text easier to inspect
pd.set_option("display.max_colwidth", None)

print("\nZERO-SHOT: BEST REVIEWS")
display(manual_zero_best)

print("\nZERO-SHOT: WORST REVIEWS")
display(manual_zero_worst)

print("\nFEW-SHOT: BEST REVIEWS")
display(manual_few_best)

print("\nFEW-SHOT: WORST REVIEWS")
display(manual_few_worst)


ZERO-SHOT: BEST REVIEWS


,review_id,rating,row_f1,row_precision,row_recall,exact_match,tp,fp,fn,text,true_labels,pred_labels,raw_pairs
0,000b48ff2d981127c7cb6b28cee74ba69a060c91d4be91bb24e584f2a11d4581,5,1.0,1.0,1.0,1,2,0,0,Awesome food and excellent service. My first choice for tacos and authentic taste,"product_quality_positive, service_positive","product_quality_positive, service_positive","[{""aspect"": ""Product Quality"", ""sentiment"": ""positive"", ""evidence"": ""Awesome food and authentic taste""}, {""aspect"": ""Service"", ""sentiment"": ""positive"", ""evidence"": ""excellent service""}]"
1,00233ad96585e69a211068cd21fceeeee3b6b5ad434c31a2231fc8665b17ed82,5,1.0,1.0,1.0,1,1,0,0,All halal menu highly recommended,product_quality_positive,product_quality_positive,"[{""aspect"": ""Product Quality"", ""sentiment"": ""positive"", ""evidence"": ""All halal menu highly recommended""}]"
2,002a13685b3e1645046f5979179f0f6eabd3610edec6f512cf01436d3ffc5288,2,1.0,1.0,1.0,1,1,0,0,Bread sucks,product_quality_negative,product_quality_negative,"[{""aspect"": ""Product Quality"", ""sentiment"": ""negative"", ""evidence"": ""Bread sucks""}]"
3,0057dba601b62128667b9bd73afdf277ed90dfe824b410c35a11f8e33c4f3957,1,1.0,1.0,1.0,1,2,0,0,"Awful,they didn't provide me a statement print that shows I've been using the metropcs for ayear (I needed it for college to show I was instated,now college cgarge me as an alien student#5times more) :(((","service_negative, general_negative","service_negative, general_negative","[{""aspect"": ""Service"", ""sentiment"": ""negative"", ""evidence"": ""they didn't provide me a statement print""}, {""aspect"": ""General"", ""sentiment"": ""negative"", ""evidence"": ""Awful, now college charge me as an alien student#5times more""}]"
4,00861e4ed9b0728bc62e80e55eba1e1c9e0a24f4477902860bc1dbf02ebdcd54,5,1.0,1.0,1.0,1,1,0,0,They always take care of my cars.,service_positive,service_positive,"[{""aspect"": ""Service"", ""sentiment"": ""positive"", ""evidence"": ""They always take care of my cars.""}]"
5,008d87805640aa2ecfc0ea5e2eedd9ef3e7b425eab7b9310a078d5a451811d88,3,1.0,1.0,1.0,1,4,0,0,Coffee was warm perfect drinking temperature one i got it. Which is fine but by the time i actually got my food it was cold. The coffee was good just wish it had been warmer. Food was also pretty tastey for the price though i could have gone to a restraunt. I will give this place another try at some point but after today i think the place across town is much better.,"product_quality_positive, product_quality_negative, price_value_positive, general_negative","product_quality_positive, product_quality_negative, price_value_positive, general_negative","[{""aspect"": ""Product Quality"", ""sentiment"": ""mixed"", ""evidence"": ""Coffee was warm perfect but food was cold""}, {""aspect"": ""Price/Value"", ""sentiment"": ""positive"", ""evidence"": ""Food was pretty tasty for the price""}, {""aspect"": ""General"", ""sentiment"": ""negative"", ""evidence"": ""place across town is much better""}]"
6,00a4e87097e38d857b6e5109516528164e8fc8e272d938edc6b865d0867adb16,1,1.0,1.0,1.0,1,4,0,0,"I was new to this place and the first time wasn’t too bad but lately they’ve been really disorganized. I placed an order a day ago and they got the whole order wrong , when I went back to tell them they weren’t too happy as if it was out fault they didn’t get the order right . They made us wait for another half hour to remake the two pizzas they messed up on and wait for third one they didn’t make . When we got the pizzas there was more dough then actual pizza . The slices on the pizza weren’t cut on one of the pizzas . Not coming back here again .","product_quality_negative, service_negative, wait_time_negative, general_negative","product_quality_negative, service_negative, wait_time_negative, general_negative","[{""aspect"": ""Service"", ""sentiment"": ""negative"", ""evidence"": ""they weren\u2019t too happy as if it was our fault they didn\u2019t get the order right""}, {""aspect"":


ZERO-SHOT: WORST REVIEWS


,review_id,rating,row_f1,row_precision,row_recall,exact_match,tp,fp,fn,text,true_labels,pred_labels,raw_pairs
0,2467735045b2685863dec079814d8c2457787c30c0df6eb23798fb5dce18a15e,4,0.0,0.0,0.0,0,0,4,2,"This is a small room on the second floor of an office building, equipped with a wall projector and desks which are a good height for performing CPR on the dummies. Despite having to bring in a certificate showing you completed the online portion, the instructor replays the Red Cross videos you already watched online, and I didn't see the point of that. The physical part that you have to do could have been completed in an hour, not two, if this was avoided. They have male and infant dummies that give feedback (a clicking sound and green light) that lets you know you are performing compressions correctly, which is helpful. The practice AEDs work well. There are places that are about $15 cheaper for BLS, but the online scheduling was fast and convenient. I got an email from the instructor a day later with the link for printing my BLS card, which worked fine.","wait_time_positive, price_value_negative","product_quality_positive, service_positive, service_negative, wait_time_negative","[{""aspect"": ""Product Quality"", ""sentiment"": ""positive"", ""evidence"": ""male and infant dummies that give feedback... which is helpful; practice AEDs work well""}, {""aspect"": ""Service"", ""sentiment"": ""mixed"", ""evidence"": ""instructor replays the Red Cross videos you already watched online (negative); email from the instructor a day later with the link for printing my BLS card, which worked fine (positive)""}, {""aspect"": ""Wait Time"", ""sentiment"": ""negative"", ""evidence"": ""physical part could have been completed in an hour, not two""}, {""aspect"": ""Price/Value"", ""sentiment"": ""neutral"", ""evidence"": ""places about $15 cheaper for BLS, but online scheduling was fast and convenient""}, {""aspect"": ""General"", ""sentiment"": ""neutral"", ""evidence"": ""small room on the second floor of an office building, equipped with a wall projector and desks""}]"
1,3eab325de2513d5b541cb948f587763f49fd1cee8b5934cd288bc94034b1e426,3,0.0,0.0,0.0,0,0,2,3,"Very nice place, very questionable people. A gamble everytime","product_quality_positive, service_negative, general_negative","atmosphere_positive, atmosphere_negative","[{""aspect"": ""Atmosphere"", ""sentiment"": ""mixed"", ""evidence"": ""Very nice place, very questionable people. A gamble everytime""}]"
2,0f8f69bcfccd024dc6cca8e4cd55dd161148361e9ca58c9ee4d39c1cfbe4c51d,4,0.0,0.0,0.0,0,0,3,1,"Has large CVS & great Ralph's. Several eateries, pizza & asian included, & even a good laundry mat! Good amount of parking & well lit. Even has bike/pedistrian trail directly behind square. I come here 2/3 times a week because of Ralph's.",atmosphere_negative,"product_quality_positive, atmosphere_positive, general_positive","[{""aspect"": ""Product Quality"", ""sentiment"": ""positive"", ""evidence"": ""great Ralph's""}, {""aspect"": ""Atmosphere"", ""sentiment"": ""positive"", ""evidence"": ""well lit, bike/pedistrian trail directly behind square""}, {""aspect"": ""General"", ""sentiment"": ""positive"", ""evidence"": ""Has large CVS & good amount of parking""}]"
3,10653e3bfe46384db2419b8a52f2e1d5f2afc98002e80fb0f3ef7afad4df0c4d,3,0.0,0.0,0.0,0,0,3,1,"First time there for holiday lunch. Adequate and fine. Tables and room seemed small for a group of over 100. Food was ok. Not enough restrooms though. We had a photo booth in the room, not sure if it was part of the gig but cool addition.",product_quality_negative,"cleanliness_negative, atmosphere_negative, general_positive","[{""aspect"": ""Product Quality"", ""sentiment"": ""neutral"", ""evidence"": ""Food was ok""}, {""aspect"": ""Cleanliness"", ""sentiment"": ""negative"", ""evidence"": ""Not enough restrooms""}, {""aspect"": ""Atmosphere"", ""sentiment"": ""negative"", ""evidence"": ""Tables and room seemed small for a group of over 100""}, {""


FEW-SHOT: BEST REVIEWS


,review_id,rating,row_f1,row_precision,row_recall,exact_match,tp,fp,fn,text,true_labels,pred_labels,raw_pairs
0,000b48ff2d981127c7cb6b28cee74ba69a060c91d4be91bb24e584f2a11d4581,5,1.0,1.0,1.0,1,2,0,0,Awesome food and excellent service. My first choice for tacos and authentic taste,"product_quality_positive, service_positive","product_quality_positive, service_positive","[{""aspect"": ""Product Quality"", ""sentiment"": ""positive"", ""evidence"": ""Awesome food""}, {""aspect"": ""Service"", ""sentiment"": ""positive"", ""evidence"": ""excellent service""}]"
1,001226755d5e472487797f43f7c9415c4a4b34cc66e27e2dfcbf29223bfad21e,2,1.0,1.0,1.0,1,2,0,0,"Don't bring kids here. Cockroaches, drug dealers, tweakers, a weird lady that rides around on a little Razer skooter she's like 60. Called about a lost debit card here and was promised a call back, no call back. Soon as I got to this place , there is a woman at the counter upset because she was charged an extra day without her knowledge and was asking of she can stay for that day she was charged they refused her. And they refused to give her money back...?? Sounds like thievery","service_negative, cleanliness_negative","service_negative, cleanliness_negative","[{""aspect"": ""Cleanliness"", ""sentiment"": ""negative"", ""evidence"": ""Cockroaches""}, {""aspect"": ""Service"", ""sentiment"": ""negative"", ""evidence"": ""promised a call back, no call back; refused her; refused to give her money back""}]"
2,00233ad96585e69a211068cd21fceeeee3b6b5ad434c31a2231fc8665b17ed82,5,1.0,1.0,1.0,1,1,0,0,All halal menu highly recommended,product_quality_positive,product_quality_positive,"[{""aspect"": ""Product Quality"", ""sentiment"": ""positive"", ""evidence"": ""All halal menu highly recommended""}]"
3,0029850e3a0c2448431f18e45a874134c20c8824c254d76334aaf2cdb2c5a925,5,1.0,1.0,1.0,1,3,0,0,Good staff good food good program loved it,"product_quality_positive, service_positive, general_positive","product_quality_positive, service_positive, general_positive","[{""aspect"": ""Service"", ""sentiment"": ""positive"", ""evidence"": ""Good staff""}, {""aspect"": ""Product Quality"", ""sentiment"": ""positive"", ""evidence"": ""good food""}, {""aspect"": ""General"", ""sentiment"": ""positive"", ""evidence"": ""good program loved it""}]"
4,002a13685b3e1645046f5979179f0f6eabd3610edec6f512cf01436d3ffc5288,2,1.0,1.0,1.0,1,1,0,0,Bread sucks,product_quality_negative,product_quality_negative,"[{""aspect"": ""Product Quality"", ""sentiment"": ""negative"", ""evidence"": ""Bread sucks""}]"
5,008d87805640aa2ecfc0ea5e2eedd9ef3e7b425eab7b9310a078d5a451811d88,3,1.0,1.0,1.0,1,4,0,0,Coffee was warm perfect drinking temperature one i got it. Which is fine but by the time i actually got my food it was cold. The coffee was good just wish it had been warmer. Food was also pretty tastey for the price though i could have gone to a restraunt. I will give this place another try at some point but after today i think the place across town is much better.,"product_quality_positive, product_quality_negative, price_value_positive, general_negative","product_quality_positive, product_quality_negative, price_value_positive, general_negative","[{""aspect"": ""Product Quality"", ""sentiment"": ""mixed"", ""evidence"": ""Coffee was warm perfect... but by the time i got my food it was cold. The coffee was good... Food was also pretty tasty""}, {""aspect"": ""Price/Value"", ""sentiment"": ""positive"", ""evidence"": ""pretty tasty for the price""}, {""aspect"": ""General"", ""sentiment"": ""negative"", ""evidence"": ""place across town is much better""}]"
6,00a4e87097e38d857b6e5109516528164e8fc8e272d938edc6b865d0867adb16,1,1.0,1.0,1.0,1,4,0,0,"I was new to this place and the first time wasn’t too bad but lately they’ve been really disorganized. I placed an order a day ago and they got the whole order wrong , when I went back to tell them they weren’t too happy as if it was out fault they didn’t get the order right . They made us wait for an


FEW-SHOT: WORST REVIEWS


,review_id,rating,row_f1,row_precision,row_recall,exact_match,tp,fp,fn,text,true_labels,pred_labels,raw_pairs
0,2467735045b2685863dec079814d8c2457787c30c0df6eb23798fb5dce18a15e,4,0.0,0.0,0.0,0,0,4,2,"This is a small room on the second floor of an office building, equipped with a wall projector and desks which are a good height for performing CPR on the dummies. Despite having to bring in a certificate showing you completed the online portion, the instructor replays the Red Cross videos you already watched online, and I didn't see the point of that. The physical part that you have to do could have been completed in an hour, not two, if this was avoided. They have male and infant dummies that give feedback (a clicking sound and green light) that lets you know you are performing compressions correctly, which is helpful. The practice AEDs work well. There are places that are about $15 cheaper for BLS, but the online scheduling was fast and convenient. I got an email from the instructor a day later with the link for printing my BLS card, which worked fine.","wait_time_positive, price_value_negative","product_quality_positive, service_positive, service_negative, wait_time_negative","[{""aspect"": ""Atmosphere"", ""sentiment"": ""neutral"", ""evidence"": ""small room on the second floor of an office building""}, {""aspect"": ""Service"", ""sentiment"": ""mixed"", ""evidence"": ""instructor replays the Red Cross videos you already watched online""}, {""aspect"": ""Wait Time"", ""sentiment"": ""negative"", ""evidence"": ""physical part could have been completed in an hour, not two""}, {""aspect"": ""Product Quality"", ""sentiment"": ""positive"", ""evidence"": ""male and infant dummies that give feedback""}, {""aspect"": ""Price/Value"", ""sentiment"": ""neutral"", ""evidence"": ""places that are about $15 cheaper for BLS""}, {""aspect"": ""Service"", ""sentiment"": ""positive"", ""evidence"": ""online scheduling was fast and convenient""}, {""aspect"": ""Service"", ""sentiment"": ""positive"", ""evidence"": ""email from the instructor a day later with the link for printing my BLS card""}]"
1,3eab325de2513d5b541cb948f587763f49fd1cee8b5934cd288bc94034b1e426,3,0.0,0.0,0.0,0,0,2,3,"Very nice place, very questionable people. A gamble everytime","product_quality_positive, service_negative, general_negative","atmosphere_positive, atmosphere_negative","[{""aspect"": ""Atmosphere"", ""sentiment"": ""mixed"", ""evidence"": ""Very nice place, very questionable people""}]"
2,2df767dfbb71b27eaef51859b3622a0ab49730d311c0b70e501adf9d3e2c2d9b,1,0.0,0.0,0.0,0,0,3,1,Always closed but it's coo. I just break in and make my own dirty chais. Hella good.,service_negative,"product_quality_positive, general_positive, general_negative","[{""aspect"": ""General"", ""sentiment"": ""mixed"", ""evidence"": ""Always closed but it's coo""}, {""aspect"": ""Product Quality"", ""sentiment"": ""positive"", ""evidence"": ""Hella good""}]"
3,4dd5bfc27aba0a2e29ca7a7c34acdd8a225c95e037af3a44e56686a247acbccc,4,0.0,0.0,0.0,0,0,3,1,"It's a good sized park with no issues of homelessness around it. Dog friendly and very friendly neighborhood. Issues: Grass and weed maintenance is sometimes an issue. Kids section is little uninteresting with uneven ground in sections. However, i like to unleash my puppy there and he enjoys every bit of it. I like it way better than dog parks.",atmosphere_positive,"cleanliness_negative, atmosphere_negative, general_positive","[{""aspect"": ""General"", ""sentiment"": ""positive"", ""evidence"": ""good sized park with no issues of homelessness, dog friendly, very friendly neighborhood""}, {""aspect"": ""Cleanliness"", ""sentiment"": ""negative"", ""evidence"": ""Grass and weed maintenance is sometimes an issue""}, {""aspect"": ""Atmosphere"", ""sentiment"": ""negative"", ""evidence"": ""Kids section is little uninteresting with uneven ground""}]"
4,564300f16976d8d1d3410041044d30fc15a8ccc5a80460103622f6ac7e88001e,3,0.0,0.0,0.0,0,0,2,2,Cheap looking 